# 04 — Conversation History

This notebook covers the **history** item from the project roadmap (see README): extending notebook 03's single-shot `ask()` into a multi-turn conversation. Follow-up questions from the operator often depend on what was already asked/answered — this notebook demonstrates threading `chat_history` through both retrieval (via history-aware query reformulation) and generation.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from factory_floor.config import VECTOR_DIR, COLLECTION_NAME

if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY in a .env file before running this notebook.')


## 1. Load the existing vector database

No PDF ingestion or embedding should happen here. That work was already completed in notebook 02 and cached in Chroma.

In [2]:
from factory_floor.vectorstore import get_embeddings, load_vectorstore
from factory_floor.rag import build_retriever

embeddings = get_embeddings()
vectorstore = load_vectorstore(VECTOR_DIR, COLLECTION_NAME, embeddings=embeddings)
retriever = build_retriever(vectorstore, k=5)
print('Persistent retriever loaded.')


Persistent retriever loaded.


## 2. Why follow-up questions need history-aware retrieval

A follow-up like *"what if that fault code has already cleared?"* means nothing to a vector search on its own — the embedding of "that fault code" carries no equipment-specific signal. `factory_floor.rag.contextualize_question` rewrites the follow-up into a standalone question (using the conversation so far) **before** retrieval runs, so the search actually has something specific to match against. This only costs an extra LLM call on turns after the first — the very first question in a conversation is never rewritten.

In [3]:
from factory_floor.rag import ask, build_chat_history, get_llm

llm = get_llm()
turns = []


## 3. Turn 1 — initial question

In [4]:
question_1 = 'A SINAMICS G120 drive is repeatedly tripping. What should a technician inspect before deciding on a cause?'

turns.append(ask(question_1, retriever, llm, chat_history=build_chat_history(turns)))

print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


For a SINAMICS G120 drive that is repeatedly tripping, a technician should inspect the following before deciding on a cause:

1. Check for DC link overvoltage conditions:
   - Verify if the motor is regenerating too much energy.
   - Measure the line supply voltage to ensure it is not too high.
   - Check for any line phase interruptions.
   - Confirm that the DC link voltage controller is activated (parameters p1240, p1280).
   - Review the dynamic response of the DC link voltage controller.
   - Consider increasing ramp-down time (p1121) and setting rounding times (p1130, p1136) to relieve the DC link voltage controller, especially in U/f operation [SOURCE 1].

2. Inspect the line supply and infeed:
   - Check the line supply voltage at the input terminals.
   - Verify the line supply voltage setting (p0210).
   - Ensure precharging resistors have cooled down and observe permissible precharging frequency.
   - Check the DC link capacitance and reduce if exceeding maximum permissible 

## 4. Turn 2 — an elliptical follow-up

The next question only makes sense in light of turn 1 — it never repeats "SINAMICS G120" or "tripping". We print the rewritten `standalone_question` so the reformulation step is visible, not just trusted.

In [5]:
question_2 = 'What if that fault code has already cleared by the time I check?'

turns.append(ask(question_2, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
If the fault code on the SINAMICS G120 drive has already cleared by the time you check, what diagnostic steps or tools can you use to retrieve historical fault information or analyze the cause of the previous trip?

ANSWER:
If the fault code has already cleared by the time you check, the documentation suggests the following approach:

- Review the fault history or event log if available to identify the fault code and its timing.
- For faults related to safety parameters or commissioning (e.g., fault values 130, 1000, 2000, 2003), carry out the safety commissioning routine again and verify safety parameter settings (p9601, p9801, p9602, p9802, p9799, p9899) as applicable [SOURCES 1,2].
- For hardware or software malfunctions, perform a power cycle of the relevant component and monitor if the fault recurs [SOURCE 3].
- Check line supply and wiring for transient faults that may have caused the trip but cleared (e.g., phase failure, voltage dips)

## 5. Turn 3 — a second follow-up

One more hop, to confirm the conversation keeps working beyond a single follow-up.

In [6]:
question_3 = 'And which of those checks needs the drive powered down first?'

turns.append(ask(question_3, retriever, llm, chat_history=build_chat_history(turns)))

print('STANDALONE QUESTION (rewritten by the LLM):')
print(turns[-1]['standalone_question'])
print('\nANSWER:')
print(turns[-1]['answer'])
print('\nSOURCES USED')
print(turns[-1]['sources'])


STANDALONE QUESTION (rewritten by the LLM):
Which specific inspections or checks related to troubleshooting a SINAMICS G120 drive that has tripped—such as verifying line supply wiring, DRIVE-CLiQ communication cables, braking module connections, or adjusting safety parameters—require the drive to be powered down before performing them?

ANSWER:
The checks that require the drive to be powered down first include:

- Waiting until the precharging resistors have cooled down, which preferably involves disconnecting the infeed unit from the line supply (power down the drive) before reconnecting to avoid damage or faults [SOURCES 2,4].

- Checking or reducing the DC link capacitance, which may require powering down the drive to safely access and modify hardware components [SOURCES 2,4].

- Inspecting or repairing internal wiring such as DRIVE-CLiQ cables or braking module connections, which generally requires the drive to be powered off to ensure safety and prevent damage [SOURCES 1,3].

Othe

## Milestone checkpoint

The pipeline is now:

`follow-up question → history-aware reformulation → retrieval → contextualized answer`

This is exactly what the Streamlit app's follow-up box (rendered below each answer) exposes to the operator. Per the README roadmap, **history** is done — vision, agents, memory and safety validation remain.